In [20]:
import sys
import pandas as pd
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

import torch
import torch.nn as nn
from torchvision import models
from src.data.dataloader import MIDASDataset
from torchvision import transforms
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from torchvision.models import resnet50, ResNet50_Weights

In [21]:
transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])
metadata = pd.read_csv("/Users/oliviagleason/Documents/year three/term nine/dsci 410L/project/midas.csv")

train_ids, test_ids = train_test_split(metadata["midas_record_id"].unique(), test_size=0.15, random_state=42)
train_ids, val_ids = train_test_split(train_ids, test_size=0.1765, random_state=42)

train = metadata[metadata["midas_record_id"].isin(train_ids)]
val = metadata[metadata["midas_record_id"].isin(val_ids)]
test = metadata[metadata["midas_record_id"].isin(test_ids)]
train.to_csv("train.csv", index=False)
val.to_csv("val.csv", index=False)
test.to_csv("test.csv", index=False)

train_dataset = MIDASDataset(data_dir="/Users/oliviagleason/Documents/year three/term nine/dsci 410L/project/MIDAS", 
                       csv_file="train.csv", transform=transform)
val_dataset = MIDASDataset(data_dir="/Users/oliviagleason/Documents/year three/term nine/dsci 410L/project/MIDAS", 
                       csv_file="val.csv", transform=transform)
test_dataset = MIDASDataset(data_dir="/Users/oliviagleason/Documents/year three/term nine/dsci 410L/project/MIDAS", 
                       csv_file="test.csv", transform=transform)

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=True)

In [25]:
class MIDASModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.derm_encoder = models.resnet18(pretrained=True)
        derm_features = self.derm_encoder.fc.in_features
        self.derm_encoder.fc = nn.Identity()

        self.phone_encoder = models.resnet18(pretrained=True)
        phone_features = self.phone_encoder.fc.in_features
        self.phone_encoder.fc = nn.Identity()

        self.metadata_encoder = nn.Sequential(
            nn.Linear(2, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 32),
            nn.ReLU())

        combined_size = derm_features + phone_features + 32

        self.classifier = nn.Sequential(
            nn.Linear(combined_size, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1))

    def forward(self, images, image_types, metadata):
        device = images.device
        
        derm_mask = image_types == 0
        phone_mask = ((image_types == 1) | (image_types == 2) | (image_types == 3))

        if derm_mask.any():
            derm_images = images[derm_mask]
            derm_features = self.derm_encoder(derm_images)
            derm_features = derm_features.mean(dim=0)
        else:
            derm_features = torch.zeros(self.derm_encoder.fc.in_features if hasattr(self.derm_encoder.fc, "in_features")
                else 512, device=device)

        if phone_mask.any():
            phone_images = images[phone_mask]
            phone_features = self.phone_encoder(phone_images)
            phone_features = phone_features.mean(dim=0)
        else:
            phone_features = torch.zeros(self.phone_encoder.fc.in_features if hasattr(self.phone_encoder.fc, "in_features")
                else 512, device=device)

        metadata_features = self.metadata_encoder(metadata)

        combined = torch.cat([derm_features, phone_features, metadata_features], dim=0)
        
        output = self.classifier(combined)
        return output.view(1)

In [26]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MIDASModel().to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

num_epochs = 5
for epoch in range(num_epochs):

    model.train()
    running_loss = 0

    for images, image_types, metadata, label in train_loader:
        images = images[0].to(device)
        image_types = image_types[0].to(device)

        metadata = metadata[0].to(device)

        label = label.to(device).float()

        optimizer.zero_grad()

        output = model(images, image_types, metadata)

        loss = criterion(output, label)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Epoch {epoch+1} | "f"Loss: {running_loss/len(train_loader):.4f}")

/opt/anaconda3/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch 1 | Loss: 0.4698
Epoch 2 | Loss: 0.4666
Epoch 3 | Loss: 0.4482
Epoch 4 | Loss: 0.4497
Epoch 5 | Loss: 0.4248


In [27]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, image_types, metadata, label in test_loader:
        
        images = images[0].to(device)
        image_types = image_types[0].to(device)

        metadata = metadata[0].to(device)

        label = label.to(device).float()

        output = model(images, image_types, metadata)

        prediction = torch.sigmoid(output)
        prediction = (prediction > 0.5).float()

        correct += (prediction == label).sum().item()
        total += 1

print(f"Test Accuracy: {correct / total:.4f}")

Test Accuracy: 0.8350
